In [ ]:
%matplotlib inline


# 31. Locally varying anisotropy

Walker Lake's high-`V` bodies bend: one global direction (N170° in topic 19) fits
some of them and crosses others. Locally varying anisotropy (LVA) gives every location its own orientation, used for
both the variogram and the search. The orientation field can come from a gridded attribute (`from_grid`, used here),
a point cloud (`from_points`) or a wireframe (`from_mesh`).


In [ ]:
import sys
from pathlib import Path

HERE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(HERE.parents[1]))

In [ ]:
import ceres as cs
import matplotlib.pyplot as plt
import numpy as np
from common import ACCENT, map_axes, save
from matplotlib.colors import PowerNorm

samples = cs.datasets.walker_lake()
truth = cs.datasets.walker_lake_exhaustive()["V"].reshape(300, 260)
xy, v = samples.coords, samples["V"]
grid = cs.BlockModel(origin=(0.5, 0.5), size=(5, 5), count=(52, 60))
nodes = grid.centroids.astype(int)
true_at_nodes = truth[nodes[:, 1] - 1, nodes[:, 0] - 1]

The global model, fitted along N170° and across it as in topic 19: two nested structures, the minor/major ratio set
by the long one.



In [ ]:
azimuth, lag, max_lag = 170.0, 10.0, 120.0
along = cs.experimental_variogram(xy, v, lag, max_lag, azimuth=azimuth).fit(
    ["spherical", "spherical"], weighting="count/gamma"
)
across = cs.experimental_variogram(xy, v, lag, max_lag, azimuth=azimuth + 90).fit(
    ["spherical", "spherical"],
    weighting="count/gamma",
    nugget=along.nugget,
    sills=[s.sill for s in along.structures],
)
model = along.with_anisotropy(
    (azimuth, 0, 0), (across.structures[-1].range / along.structures[-1].range, 1.0)
)
print(model)

A quick isotropic estimate outlines the bodies; the gradient of that map gives the orientation of least change.
The structure tensor sums gradients over 3 cells each side, and smoothing averages orientations within 25 m.



In [ ]:
isotropic = cs.Variogram([("spherical", model.sill - model.nugget, 30.0)], nugget=model.nugget)
guide = cs.OrdinaryKriging(isotropic, cs.Search(radius=60, max_samples=16)).fit(xy, v).predict(grid)
lva = cs.LocalAnisotropy.from_grid(
    grid.with_column("guide", guide), "guide", window=3, ratios=(0.3, 1.0)
).smooth(25.0)
print(lva)
print("azimuth percentiles (10, 50, 90):", np.percentile(lva.angles[:, 0] % 180, [10, 50, 90]).round())

In [ ]:
shape = (60, 52)
extent = (0.5, 260.5, 0.5, 300.5)
fig, ax = plt.subplots(figsize=(6.4, 6), layout="constrained")
ax.imshow(
    guide.reshape(shape),
    origin="lower",
    extent=extent,
    cmap="cividis",
    norm=PowerNorm(0.5, vmin=0, vmax=1200),
)
every = (nodes[:, 0] % 15 == 3) & (nodes[:, 1] % 15 == 3)
cs.plot.directions(lva.at(grid.centroids[every]), ax=ax, scale=35, width=0.004, color=ACCENT)
map_axes(ax, "Local major direction over the guide estimate")
save(fig, "field")

Same model, same search; only the orientation changes. Errors are against the exhaustive values at the nodes.



In [ ]:
search = cs.Search(radius=100, max_samples=24, min_samples=4)
ok = cs.OrdinaryKriging(model, search).fit(xy, v)
global_estimate = ok.predict(grid)
local_estimate = ok.predict(grid, anisotropy=lva)
for name, estimate in ((f"global N{model.rotation[0]:.0f}°", global_estimate), ("local", local_estimate)):
    error = estimate - true_at_nodes
    print(
        f"{name:>13}: RMSE {np.sqrt(np.mean(error**2)):.1f} ppm, correlation {np.corrcoef(estimate, true_at_nodes)[0, 1]:.3f}"
    )

In [ ]:
norm = PowerNorm(0.5, vmin=0, vmax=1500)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6), layout="constrained")
for ax, image, title in (
    (axes[0], true_at_nodes, "True V at grid nodes"),
    (axes[1], global_estimate, "Kriging, global anisotropy"),
    (axes[2], local_estimate, "Kriging, local anisotropy"),
):
    im = ax.imshow(image.reshape(shape), origin="lower", extent=extent, norm=norm)
    map_axes(ax, title)
fig.colorbar(im, ax=axes, shrink=0.8, label="V (ppm)")
save(fig, "kriging")

Local orientations lower the error and keep the north-eastern bodies elongated east-west, where the global
direction crosses them.



Simulation takes the same field: each node's variogram and search follow the local direction, so continuity bends
with the bodies instead of crossing them. The normal-score variogram is fitted along the major axis; the field
supplies the orientation and the ratios.



In [ ]:
weights = cs.cell_declustering(xy, v, sizes=np.arange(2.5, 102.5, 2.5)).weights
scores = cs.NormalScore().fit_transform(v, weights=weights)
gaussian = cs.experimental_variogram(xy, scores, lag, max_lag, azimuth=azimuth).fit("spherical")
sgs = cs.SGS(gaussian, cs.Search(radius=100, max_samples=24)).fit(xy, v, weights=weights)
uniform = cs.LocalAnisotropy(np.zeros((1, 3)), [model.rotation], [model.ratios])
global_real = sgs.simulate(grid, n=1, seed=11, realizations=True, anisotropy=uniform).realizations[0]
local_real = sgs.simulate(grid, n=1, seed=11, realizations=True, anisotropy=lva).realizations[0]

fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.6), layout="constrained")
for ax, image, title in (
    (axes[0], global_real, f"SGS, global N{model.rotation[0]:.0f}°"),
    (axes[1], local_real, "SGS, local anisotropy"),
):
    im = ax.imshow(image.reshape(shape), origin="lower", extent=extent, norm=norm)
    map_axes(ax, title)
fig.colorbar(im, ax=axes, shrink=0.8, label="V (ppm)")
save(fig, "simulation")